# Documentation
**Author:** Spencer Ressel

**Created:** May 26th, 2026

**Inputs:**     
* Global 2.5° x 2.5° resolution, daily timeseries in netCDF format:
    - outgoing longwave radiation (OLR) data from *Liebmann and Smith (1996)*

# Imports

In [1]:
%load_ext autoreload
%autoreload 2

# -------------------------------------------------------------------
# Project setup: ensure working directory is the thesis project root
# -------------------------------------------------------------------
import os
thesis_work_directory = os.environ["THESIS_WORK"]
system_name = os.environ["SYSTEM"]
os.chdir(f"{thesis_work_directory}/python/mjo_data_analysis/")

# -------------------------------------------------------------------
# Logging configuration
# -------------------------------------------------------------------
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

# -------------------------------------------------------------------
# Core scientific computing + data processing
# -------------------------------------------------------------------
import numpy as np
import scipy
import scipy.signal as signal
from scipy.optimize import curve_fit
from datetime import datetime

import xarray as xr
import xeofs  # EOF analysis package

import re     # regex utilities
import sys    # system-level operations

# -------------------------------------------------------------------
# Project-specific auxiliary utilities
# -------------------------------------------------------------------
from auxiliary_functions.plotting_utils import (
    modified_colormap, 
    tick_labeller
)
from auxiliary_functions import xarray_utils
from auxiliary_functions.mjo_mean_state_diagnostics import (
    remove_annual_cycle,
    lanczos_bandpass_filter,
)

# -------------------------------------------------------------------
# Plotting + visualization
# -------------------------------------------------------------------
from tqdm import tqdm

from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
from matplotlib import colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation

# -------------------------------------------------------------------
# Cartopy (mapping + geospatial utilities)
# -------------------------------------------------------------------
from cartopy import crs as ccrs
from cartopy import feature as cf
from cartopy import util as cutil
from cartopy.mpl.ticker import (
    LongitudeFormatter,
    LatitudeFormatter,
    LongitudeLocator,
    LatitudeLocator,
)

# -------------------------------------------------------------------
# Seaborn (statistical plotting)
# -------------------------------------------------------------------
import seaborn as sns

logger.info("Imports Loaded")

2026-06-16 16:55:03,275 [INFO] Imports Loaded


# Set Physical Constants and Analysis Parameters

In [2]:
# Set time bounds
TIME_MIN = '1974-06-01T00:00:00.000000000'
# TIME_MAX = '2005-12-31T00:00:00.000000000'
TIME_MAX = '2022-12-31T00:00:00.000000000'

missing_days = np.arange(np.datetime64("1978-03-17"), np.datetime64("1978-12-31"))

SAMPLING_FREQUENCY = 1

# Set latitude bounds
LATITUDE_SOUTH = -25
LATITUDE_NORTH = 25

# Set central longitude
CENTRAL_LONGITUDE = 160

# Set longitude bounds
LONGITUDE_MIN = 0
LONGITUDE_MAX = 360

# Cut-off periods for intraseasonal filtering
INTRASEASONAL_LOWCUT = 200
INTRASEASONAL_HIGHCUT = 20

# Seconds per day
SECONDS_PER_DAY = 24 * 3600

# Load Data

In [3]:
logger.info("Load Variables")

if system_name == "NCAR":
    top_level_data_directory = "/glade/derecho/scratch/sressel"
elif system_name == 'UW':
    top_level_data_directory = "/home/disk/eos7/sressel/research/data"

variables_to_load = [
    'Precipitation',
    'Outgoing Longwave Radiation',
    'Zonal Wind',
    'Meridional Wind'
]

# TRMM Precipitation
if 'Precipitation' in variables_to_load:
    logger.info("    Precipitation...")
    data_directory_precipitation = "NASA/TRMM"
    file_name_precipitation = "trmm_precipitation_daily_1998_2018.nc"
    data_precipitation = xr.open_dataset(
        f"{top_level_data_directory}/{data_directory_precipitation}/{file_name_precipitation}",
        engine="netcdf4"
    )
    precipitation = data_precipitation['precipitation'].sortby('lat')

# NASA OLR (Liebmann and Smith 1996)
if 'Outgoing Longwave Radiation' in variables_to_load:
    logger.info("    Outgoing Longwave Radiation...")
    data_directory_olr = r"NOAA"
    file_name_olr = "olr.day.mean.nc"
    data_olr = xr.open_dataset(
        f"{top_level_data_directory}/{data_directory_olr}/{file_name_olr}",
        engine="netcdf4")
    outgoing_longwave_radiation = data_olr['olr'].sortby('lat')

# ERA5 Zonal Wind
if 'Zonal Wind' in variables_to_load:
    logger.info("    Zonal Wind...")
    data_directory_wind = r"ECMWF/ERA5/daily_data/"
    file_name_zonal_wind = "daily_25_degree_zonal_wind_1980_2022.nc"
    data_zonal_wind = xr.open_dataset(
        f"{top_level_data_directory}/{data_directory_wind}/{file_name_zonal_wind}",
        engine="netcdf4"
    )

    zonal_wind = data_zonal_wind["u"].sortby('lat')
    zonal_wind_longitudes = zonal_wind["lon"].values
    zonal_wind_longitudes[zonal_wind_longitudes < 0] += 360
    zonal_wind["lon"] = zonal_wind_longitudes
    zonal_wind = zonal_wind.sortby(zonal_wind.lon)

    upper_level_zonal_wind = zonal_wind.sel(plev=200)
    lower_level_zonal_wind = zonal_wind.sel(plev=850)

# ERA5 Meridional Wind
if 'Meridional Wind' in variables_to_load:
    logger.info("    Meridional Wind...")
    file_name_meridional_wind = (
        "daily_25_degree_meridional_wind_1980_2018.nc"
    )
    data_meridional_wind = xr.open_dataset(
        f"{top_level_data_directory}/{data_directory_wind}/{file_name_meridional_wind}",
        engine="netcdf4"
    )

    meridional_wind = data_meridional_wind["v"].sortby('lat')
    meridional_wind_longitudes = meridional_wind["lon"].values
    meridional_wind_longitudes[meridional_wind_longitudes < 0] += 360
    meridional_wind["lon"] = meridional_wind_longitudes
    meridional_wind = meridional_wind.sortby(meridional_wind.lon)

    upper_level_meridional_wind = meridional_wind.sel(plev=200)
    lower_level_meridional_wind = meridional_wind.sel(plev=850)

time = outgoing_longwave_radiation.time
latitude = outgoing_longwave_radiation.lat
longitude = outgoing_longwave_radiation.lon

variables_dict = {
    'Precipitation' : precipitation,
    'Outgoing Longwave Radiation' : outgoing_longwave_radiation,
    # 'Zonal Wind': zonal_wind,
    # 'Meridional Wind': meridional_wind,
    # 'Upper Level Zonal Wind' : upper_level_zonal_wind,
    # 'Lower Level Zonal Wind' : lower_level_zonal_wind,
    # 'Upper Level Meridional Wind' : upper_level_meridional_wind,
    # 'Lower Level Meridional Wind' : lower_level_meridional_wind,
}
logger.info("Finished")

2026-06-16 16:55:03,441 [INFO] Load Variables
2026-06-16 16:55:03,442 [INFO]     Precipitation...
2026-06-16 16:55:07,386 [INFO]     Outgoing Longwave Radiation...
2026-06-16 16:55:07,541 [INFO]     Zonal Wind...
2026-06-16 16:55:07,621 [INFO]     Meridional Wind...
2026-06-16 16:55:07,654 [INFO] Finished


# Process Data

Detrend the data, remove the annual cycle and the first three harmonics (seasonal cycle), and filter the data on intraseasonal timescales

## Subset Data
Specifically select the data from times of interest and from tropical latitudes

In [4]:
logger.info("Subset variables")
variables_subset = {}
for variable in variables_dict:
    logger.info(f"    {variable}...")
    if variable != 'Precipitation':
        variables_subset[variable] = variables_dict[variable].copy(deep=True)
        variables_subset[variable] = variables_dict[variable].sel(
            time=slice(TIME_MIN, TIME_MAX),
            lat=slice(LATITUDE_SOUTH, LATITUDE_NORTH)
        )

if 'Precipitation' in variables_dict:
    variables_subset['Precipitation'] = variables_dict['Precipitation'].copy(deep=True).sel(
        time=slice('1999-01-01T00:00:00.000000000', '2018-12-31T00:00:00.000000000'),
        lat=slice(LATITUDE_SOUTH, LATITUDE_NORTH)
    )

latitude = latitude.sel(lat=slice(LATITUDE_SOUTH, LATITUDE_NORTH))
time = time.sel(time=slice(TIME_MIN, TIME_MAX))

logger.info("Finished")

2026-06-16 16:55:07,801 [INFO] Subset variables
2026-06-16 16:55:07,803 [INFO]     Precipitation...
2026-06-16 16:55:07,805 [INFO]     Outgoing Longwave Radiation...
2026-06-16 16:55:07,811 [INFO] Finished


## Remove mean & Detrend the data

In [5]:
# variables_detrended = {}

# logger.info("Detrend data")
# for variable_name, variable_data in variables_subset.items():
#     logger.info(f"    {variable_name}...")
#     variables_detrended[variable_name] = xr.zeros_like(variable_data)
#     variables_detrended[variable_name][:] = signal.detrend(
#         variable_data.stats.standardize(dim='time'), 
#         axis=variable_data.get_axis_num('time'), 
#         type='linear'
#     )
# logger.info("Finished")

## Remove the Annual Cycle

In [6]:
variables_deannualized = {}
variables_annual_cycle = {}
logger.info("Remove the annual cycle")

for variable_name, variable_data in variables_subset.items():
    logger.info(f"→ {variable_name}...")
    [           
        variables_deannualized[variable_name],
        variables_annual_cycle[variable_name],
    ] = remove_annual_cycle(variable_data)    

logger.info("Finished")

2026-06-16 16:55:08,089 [INFO] Remove the annual cycle
2026-06-16 16:55:08,111 [INFO] → Outgoing Longwave Radiation...
2026-06-16 16:55:27,202 [INFO] → Precipitation...
2026-06-16 16:55:29,029 [INFO] Finished


## Filter Data 

Temporally filter the data on intraseasonal (20-100 day) timescales, using a Lanczos filter

In [7]:
nyq = 0.5
filter_order = 4
low = (1/INTRASEASONAL_LOWCUT) / nyq
high = (1/INTRASEASONAL_HIGHCUT) / nyq
b, a = signal.butter(filter_order, [low, high], btype="band")

variables_filtered = {}

logger.info(f"Filter on {INTRASEASONAL_HIGHCUT}-{INTRASEASONAL_LOWCUT} day timescales")
for variable_name, variable_data in variables_deannualized.items():
    logger.info(f"→ {variable_name}...")
    variables_filtered[variable_name] = variables_deannualized[variable_name].copy(deep=True)
    variables_filtered[variable_name].values = lanczos_bandpass_filter(
    variable_data,
    lowcut=(1 / INTRASEASONAL_LOWCUT),
    highcut=(1 / INTRASEASONAL_HIGHCUT),
    fs=SAMPLING_FREQUENCY,
    filter_axis=0,
    order=241
)
    variables_filtered[variable_name] = variables_filtered[variable_name].isel(time=slice(120, -120))

variables_filtered['Outgoing Longwave Radiation'] = variables_filtered['Outgoing Longwave Radiation'].drop_sel(time=missing_days)
logger.info("Finished")

2026-06-16 16:55:29,070 [INFO] Filter on 20-200 day timescales
2026-06-16 16:55:29,077 [INFO] → Outgoing Longwave Radiation...


2026-06-16 16:55:32,072 [INFO] → Precipitation...
2026-06-16 16:55:33,560 [INFO] Finished


# EOF Analysis

## Compute EOFs and PCs

In [11]:
model = xeofs.single.EOF(use_coslat=True, n_modes=2)
model.fit(variables_filtered['Outgoing Longwave Radiation'].sel(lat=slice(-25,25)).dropna(dim='time'), dim="time")
EOFs = -model.components()
principle_components = -model.scores()/model.scores().std(dim='time')
lag_days = np.arange(-45, 46)

## Plot EOF structures

### Plot EOFs

In [ ]:
# Set plotting parameters
output_directory = "output/mjo-compositing/"
plt.style.use('default')
plt.rcParams.update({'font.size':24})
cmap_modified = modified_colormap('coolwarm', 'white', 0.1, 0.1)
# cmap_modified = 'BrBG'
coastline_width = 1

fig = plt.figure(figsize=(16,4))
gs = GridSpec(2, 2, height_ratios=[30, 1], figure=fig)
gs.update(top=1, bottom=0, left=0, right=1, hspace=0.3, wspace=0.15)

proj = ccrs.PlateCarree(central_longitude=-180)
data_crs = ccrs.PlateCarree()

ax = [
    fig.add_subplot(gs[0,0], projection=proj),
    fig.add_subplot(gs[0,1], projection=proj)
]

cb_ax = fig.add_subplot(gs[1, :])

# Add cyclic point
cdata = xarray_utils.add_cyclic_point(
    model.scores().std(dim='time')*EOFs,
    dim='lon'
)

for index, mode in enumerate(EOFs.mode):
    # Plot data
    im = ax[index].contourf(
        cdata.lon, 
        cdata.lat, 
        cdata.sel(mode=mode), 
        transform=data_crs,
        cmap=cmap_modified,
        norm=mcolors.CenteredNorm(),
        levels=np.arange(-9, 9+6, 6),
        extend='both'
    )

    ax[index].contour(
        cdata.lon, 
        cdata.lat, 
        cdata.sel(mode=mode), 
        transform=data_crs,
        colors='k',
        levels=np.arange(-18, 18+3, 3)[np.arange(-18, 18+3, 3) != 0],
    )

    # Add colorbar
    cbar = fig.colorbar(im, cax=cb_ax, orientation='horizontal')
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label(r'W m$^{-2}$')

    ax[index].set_aspect('auto')
    ax[index].set_xlabel('')
    # ax.set_global()
    ax[index].add_feature(cf.COASTLINE, lw=coastline_width)

    gl = ax[index].gridlines(
        crs=proj,
        draw_labels=True,
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,30))
    # gl.xlocator = LongitudeLocator(30)
    gl.xformatter = LongitudeFormatter()
    gl.xlabel_style = {'fontsize':20}
    gl.ylocator = mticker.FixedLocator(np.arange(-40,40,10))
    gl.yformatter = LatitudeFormatter()
    gl.ylabel_style = {'fontsize':20}

plt.show()
# plt.savefig(f"{output_directory}/time_mean_OLR_zonal_wind_anomalies.png", dpi=300, bbox_inches='tight')

### Plot PCs

In [ ]:
# Configure plot
plt.style.use('default')
[fig, ax] = plt.subplots(1, 2, figsize=(12,6))
# plt.xlabel("RMM1")
# plt.ylabel("RMM2")

# Plot index points
colormap = sns.color_palette("viridis", as_cmap=True)

# start_time = '1992-02-12' 
# end_time = '1992-05-12'

# start_time = '1992-08-14' 
# end_time = '1992-11-12'

# start_time = np.datetime64("1975-04-01T00:00:00.000000000")
# end_time = np.datetime64("1975-04-30T00:00:00.000000000")

# start_time = '1975-02-24T00:00:00.000000000'
# end_time = '1975-04-11T00:00:00.000000000'

start_time="2020-12-29T00:00:00.000000000"
end_time="2021-02-27T00:00:00.000000000"

# start_time = primary_event_start_times[-1].values
# end_time = primary_event_end_times[-1].values

# for index, (start_time, end_time) in enumerate(zip(('1992-02-12', '1992-08-14'), ('1992-05-12', '1992-11-12'))):
for index, (start_time, end_time) in enumerate(zip((start_time, '1992-08-14'), (end_time, '1992-11-12'))):

    # Plot start_time as empty circle
    ax[index].plot(
        principle_components.sel(mode=1, time=start_time),
        principle_components.sel(mode=2, time=start_time),
        color="black",
        marker="o",
        markerfacecolor='None',
        ls="-",
        ms=15  
    )

    # Plot halfway point as filled diamond
    halfway_time = principle_components.sel(time=slice(start_time, end_time)).time.values[len(principle_components.sel(time=slice(start_time, end_time)).time) // 2]
    ax[index].plot(
        principle_components.sel(mode=1, time=halfway_time),
        principle_components.sel(mode=2, time=halfway_time),
        color="black",
        marker="D",
        markerfacecolor='black',
        ls="-",
        ms=15
    )

    ax[index].plot(
        principle_components.sel(mode=1, time=end_time),
        principle_components.sel(mode=2, time=end_time),
        color="black",
        marker="s",
        ls="-",
        ms=15    
    )
    # for time in principle_components.time.sel(time=slice(start_time, end_time)):
    ax[index].plot(
        principle_components.sel(mode=1, time=slice(start_time, end_time)),
        principle_components.sel(mode=2, time=slice(start_time, end_time)),
        color='k',
        linestyle='-',
        marker=".",
        ms=8,
    )

    # Add phase regions overlay
    circle1 = plt.Circle((0, 0), 0.3, color="#bcbcbc", fill=False, lw=1, zorder=10)
    circle2 = plt.Circle((0, 0), 0.4, color="k", fill=False, lw=1.5, zorder=10)
    circle3 = plt.Circle((0, 0), 0.5, color="#bcbcbc", fill=False, lw=1, zorder=10)
    ax[index].add_patch(circle1)
    ax[index].add_patch(circle2)
    ax[index].add_patch(circle3)

    ax[index].axhline(y=0, color="k", lw=1, ls="-")
    ax[index].axvline(x=0, color="k", lw=1, ls="-")

    # Add lines to differentiate the phases
    ax[index].plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax[index].plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax[index].plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax[index].plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )

    ax[index].set_xlim(-3,3)
    ax[index].set_ylim(-3,3)

    # # Add phase labels
    ax[index].text(
        2.9, 0.1,
        f'Category A',
        horizontalalignment='right',
        verticalalignment='center',
        fontsize=12
    )
    ax[index].text(
        0, 2.8,
        f'Category B',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )
    ax[index].text(
        -2.9, .1,
        f'Category C',
        horizontalalignment='left',
        verticalalignment='center',
        fontsize=12
    )
    ax[index].text(
        0, -2.8,
        f'Category D',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )

    ax[index].spines['left'].set_position('zero')
    ax[index].spines['bottom'].set_position('zero')

    # Hide the top and right spines
    ax[index].spines['right'].set_color('none')
    ax[index].spines['top'].set_color('none')

    # Ensure tick marks follow the spines to the center
    ax[index].xaxis.set_ticks_position('bottom')
    ax[index].yaxis.set_ticks_position('left')

    ax[index].set_aspect("equal")
plt.tight_layout()

## PC Lag Correlation

In [ ]:
lag_correlation = xr.DataArray(
    data = np.empty((len(lag_days))),
    dims=['lag'],
    coords={'lag': lag_days}
)
for index, lag in enumerate(lag_days):
    lag_correlation[index] = xr.dot(
        principle_components.sel(mode=1).stats.standardize(dim='time'),
        principle_components.sel(mode=2).stats.standardize(dim='time').roll(time=lag)
    )

## Assign category labels

In [12]:
def get_category(principle_components):

    if 'time' in principle_components.dims and principle_components.time.size != 1:
        raise ValueError("Input principle_components must have a single time")
    
    if np.isnan(principle_components.sel(mode=1)) or np.isnan(principle_components.sel(mode=2)):
        return 'N/A'
    elif np.sqrt(principle_components.sel(mode=1)**2 + principle_components.sel(mode=2)**2) <= 0.3:
        return 'N'
    elif (principle_components.sel(mode=1) > 0) and (principle_components.sel(mode=1) > np.abs(principle_components.sel(mode=2))):
        return 'A'
    elif (principle_components.sel(mode=2) > 0) and (principle_components.sel(mode=2) > np.abs(principle_components.sel(mode=1))):
        return 'B'
    elif (principle_components.sel(mode=1) < 0) and (-principle_components.sel(mode=1) > np.abs(principle_components.sel(mode=2))):
        return 'C'
    elif (principle_components.sel(mode=2) < 0) and(-principle_components.sel(mode=2) > np.abs(principle_components.sel(mode=1))):
        return 'D'


In [13]:
import itertools

amplitude = np.sqrt(principle_components.sel(mode=1)**2 + principle_components.sel(mode=2)**2)

start_time = principle_components.isel(time=0).time.values
end_time = principle_components.isel(time=-1).time.values

time_range = np.arange(start_time, end_time, np.timedelta64(1, 'D'))

category = []
category.append(get_category(principle_components.sel(time=start_time)))

for i, t in enumerate(time_range):
    if t == start_time:
        continue

    if t-np.timedelta64(1, 'D') not in principle_components.time or t not in principle_components.time:
        category.append('E')
        continue

    previous_category = get_category(principle_components.sel(time=t-np.timedelta64(1, 'D')))
    nominal_category = get_category(principle_components.sel(time=t))

    if amplitude.sel(time=t) < 0.3:
        category.append('N')
    elif amplitude.sel(time=t) > 0.5:
        category.append(nominal_category)
    else:
        if previous_category == 'N':
            category.append('N')
        else:
            category.append(nominal_category)

categories = xr.DataArray(
    data=category,
    dims=['time'],
    coords={'time':time_range}
)

## Identify MJO events

### Find Primary & Secondary Events

In [14]:
categories_string = "".join(categories.values.astype(str))

# Primary events
# primary_event_pattern = r"N+A+B+C+D+"
primary_event_pattern = r"N+A+B+C+"
primary_event_matches = list(re.finditer(primary_event_pattern, categories_string))

primary_event_start_indices = [m.start() for m in primary_event_matches]
primary_event_start_times = categories.time[primary_event_start_indices]

primary_event_end_indices = [m.end()-1 for m in primary_event_matches]
primary_event_end_times = categories.time[primary_event_end_indices]

primary_event_max_times = categories.sel(time=[principle_components.where(categories == 'A', drop=True).sel(time=slice(primary_event_start_times[i], primary_event_end_times[i]), mode=1).idxmax().values for i in range(len(primary_event_start_times))]).time


# Secondary events
secondary_event_pattern = r"D+A+B+C+D+"
secondary_event_matches = list(re.finditer(secondary_event_pattern, categories_string))

secondary_event_start_indices = [m.start() for m in secondary_event_matches]
secondary_event_start_times = categories.time[secondary_event_start_indices]

secondary_event_end_indices = [m.end() for m in secondary_event_matches]
secondary_event_end_times = categories.time[secondary_event_end_indices]

secondary_event_max_times = categories.sel(time=[principle_components.where(categories == 'A', drop=True).sel(time=slice(secondary_event_start_times[i], secondary_event_end_times[i]), mode=1).idxmax().values for i in range(len(secondary_event_start_times))]).time

In [17]:
[print(primary_event_start_times.isel(time=i).values, primary_event_end_times.isel(time=i).values) for i in range(len(primary_event_end_times.time))]

1975-02-24T00:00:00.000000000 1975-03-31T00:00:00.000000000
1975-11-23T00:00:00.000000000 1975-12-27T00:00:00.000000000
1980-09-27T00:00:00.000000000 1980-10-31T00:00:00.000000000
1981-07-11T00:00:00.000000000 1981-08-12T00:00:00.000000000
1981-11-13T00:00:00.000000000 1981-12-17T00:00:00.000000000
1982-12-14T00:00:00.000000000 1983-01-15T00:00:00.000000000
1983-10-29T00:00:00.000000000 1983-12-13T00:00:00.000000000
1984-09-18T00:00:00.000000000 1984-10-27T00:00:00.000000000
1986-10-17T00:00:00.000000000 1986-11-21T00:00:00.000000000
1987-11-28T00:00:00.000000000 1987-12-25T00:00:00.000000000
1988-10-30T00:00:00.000000000 1988-12-25T00:00:00.000000000
1989-06-24T00:00:00.000000000 1989-07-25T00:00:00.000000000
1990-07-15T00:00:00.000000000 1990-08-24T00:00:00.000000000
1990-09-12T00:00:00.000000000 1990-10-10T00:00:00.000000000
1992-09-19T00:00:00.000000000 1992-11-04T00:00:00.000000000
1992-11-17T00:00:00.000000000 1992-12-28T00:00:00.000000000
1993-11-21T00:00:00.000000000 1994-01-03

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [10]:
primary_event_end_times

NameError: name 'primary_event_end_times' is not defined

#### Specific MJO event PC timeseries

In [ ]:
# Configure plot
plt.style.use('default')

# Plot index points
colormap = sns.color_palette("viridis", as_cmap=True)

start_time_list = [
    # primary_event_start_times[-3].values,
    primary_event_start_times[-2].values,
    # primary_event_start_times[-1].values,
]
end_time_list = [
    # primary_event_end_times[-3].values,
    primary_event_end_times[-2].values,
    # primary_event_end_times[-1].values,
]


# for index, (start_time, end_time) in enumerate(zip(('1992-02-12', '1992-08-14'), ('1992-05-12', '1992-11-12'))):
for index, (start_time, end_time) in enumerate(zip(start_time_list, end_time_list)):

    [fig, ax] = plt.subplots(1, 1, figsize=(12,6))

    ax.set_title(f"Event starting {start_time.astype('M8[ms]').astype('O').strftime('%d%^b%Y')}", fontsize=14, pad=10)
    # Plot start_time as empty circle
    ax.plot(
        principle_components.sel(mode=1, time=start_time),
        principle_components.sel(mode=2, time=start_time),
        color="black",
        marker="o",
        markerfacecolor='None',
        ls="-",
        ms=15  
    )

    # Plot halfway point as filled diamond
    halfway_time = principle_components.sel(time=slice(start_time, end_time)).time.values[len(principle_components.sel(time=slice(start_time, end_time)).time) // 2]
    ax.plot(
        principle_components.sel(mode=1, time=halfway_time),
        principle_components.sel(mode=2, time=halfway_time),
        color="black",
        marker="D",
        markerfacecolor='black',
        ls="-",
        ms=15
    )

    ax.plot(
        principle_components.sel(mode=1, time=end_time),
        principle_components.sel(mode=2, time=end_time),
        color="black",
        marker="s",
        ls="-",
        ms=15    
    )
    # for time in principle_components.time.sel(time=slice(start_time, end_time)):
    ax.plot(
        principle_components.sel(mode=1, time=slice(start_time, end_time)),
        principle_components.sel(mode=2, time=slice(start_time, end_time)),
        color='k',
        linestyle='-',
        marker=".",
        ms=8,
    )

    # Add phase regions overlay
    circle1 = plt.Circle((0, 0), 0.3, color="#bcbcbc", fill=False, lw=1, zorder=10)
    circle2 = plt.Circle((0, 0), 0.4, color="k", fill=False, lw=1.5, zorder=10)
    circle3 = plt.Circle((0, 0), 0.5, color="#bcbcbc", fill=False, lw=1, zorder=10)
    ax.add_patch(circle1)
    ax.add_patch(circle2)
    ax.add_patch(circle3)

    ax.axhline(y=0, color="k", lw=1, ls="-")
    ax.axvline(x=0, color="k", lw=1, ls="-")

    # Add lines to differentiate the phases
    ax.plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax.plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax.plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax.plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )

    ax.set_xlim(-3,3)
    ax.set_ylim(-3,3)

    # # Add phase labels
    ax.text(
        2.9, 0.1,
        f'Category A',
        horizontalalignment='right',
        verticalalignment='center',
        fontsize=12
    )
    ax.text(
        0, 2.8,
        f'Category B',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )
    ax.text(
        -2.9, .1,
        f'Category C',
        horizontalalignment='left',
        verticalalignment='center',
        fontsize=12
    )
    ax.text(
        0, -2.8,
        f'Category D',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )

    ax.spines['left'].set_position('zero')
    ax.spines['bottom'].set_position('zero')

    # Hide the top and right spines
    ax.spines['right'].set_color('none')
    ax.spines['top'].set_color('none')

    # Ensure tick marks follow the spines to the center
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')

    ax.set_aspect("equal")
plt.tight_layout()

#### Specific MJO event horizontal structures

In [ ]:
event = -2
start_time = primary_event_start_times[event].values
max_time = primary_event_max_times[event].values
end_time = primary_event_end_times[event].values

times_to_plot = np.arange(
    np.datetime64(start_time) - np.timedelta64(1, 'D'),
    np.datetime64(end_time) + np.timedelta64(5, 'D'),
    np.timedelta64(5, 'D')
    )

fig = plt.figure(figsize=(12, len(times_to_plot)))
gs = GridSpec(len(times_to_plot), 2, figure=fig, width_ratios=[30, 1])
gs.update(top=1, bottom=0, left=0, right=0.8, hspace=0.1, wspace=0.05)

proj = ccrs.PlateCarree(central_longitude=-180)
data_crs = ccrs.PlateCarree()

ax = [
    fig.add_subplot(gs[i, 0], projection=proj) 
    for i in range(len(times_to_plot))
]

cbar_axis = fig.add_subplot(gs[:, 1])

cdata = xarray_utils.add_cyclic_point(
    variables_filtered['Outgoing Longwave Radiation'],
    dim='lon'
)

ax[0].set_title(f"Event starting {start_time.astype('M8[ms]').astype('O').strftime('%d%^b%Y')}", fontsize=16)
for index, time in enumerate(times_to_plot):
    im = ax[index].contourf(
        cdata.lon,
        cdata.lat,
        cdata.sel(time=time),
        levels=np.arange(-125, 125, 25),
        transform=data_crs,
        cmap='coolwarm',
        norm=mcolors.CenteredNorm(vcenter=0),
    )
    ax[index].add_feature(cf.COASTLINE, lw=1)
    ax[index].text(
        s=f"{(time-primary_event_max_times[event]).values.astype('timedelta64[D]')}", 
        x=1-0.015, 
        y=0.9,
        transform=ax[index].transAxes,
        fontsize=12,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none')
        )

    arrow_spacing = 2
    # ax[index].quiver(
    #     variables_filtered['Zonal Wind'].lon[::2*arrow_spacing],
    #     variables_filtered['Zonal Wind'].lat[::arrow_spacing],
    #     variables_filtered['Zonal Wind'].sel(plev=850, time=time)[::arrow_spacing, ::2*arrow_spacing].values,
    #     variables_filtered['Meridional Wind'].sel(plev=850, time=time)[::arrow_spacing, ::2*arrow_spacing].values,
    #     transform=data_crs,
    #     width=0.002,
    #     scale=200
    # )

    gl = ax[index].gridlines(
        crs=proj,
        draw_labels=(True if index == len(times_to_plot) - 1 else False),
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,60))
    gl.xformatter = LongitudeFormatter()
    gl.ylocator = mticker.FixedLocator(np.arange(-30,45,15))
    gl.yformatter = LatitudeFormatter()

fig.colorbar(im, cax=cbar_axis, orientation='vertical', label=r'W m$^{-2}$')

plt.show()

### Composite over Lag Day

In [ ]:
compositing_time = slice("2018-01-01", "2022-12-31")
primary_da = []
primary_pc = []
for time in primary_event_max_times:

    primary_da.append(
        variables_filtered['Outgoing Longwave Radiation'].sel(
            time=slice(
                primary_event_max_times.sel(time=time)-np.timedelta64(45, 'D'),
                primary_event_max_times.sel(time=time)+np.timedelta64(45, 'D')
            )
        ).assign_coords(time=lag_days).rename(time='lag').expand_dims('event', axis=0).assign_coords(
            timestamp=("event", [primary_event_max_times.sel(time=time).values])
        )
    )
    primary_pc.append(
        principle_components.sel(
            time=slice(
                primary_event_max_times.sel(time=time)-np.timedelta64(45, 'D'),
                primary_event_max_times.sel(time=time)+np.timedelta64(45, 'D')
            )
        ).assign_coords(time=lag_days).rename(time='lag').expand_dims('event', axis=0).assign_coords(
            timestamp=("event", [primary_event_max_times.sel(time=time).values])
        )
    )
primary_samples = xr.concat(primary_da, dim='event')
primary_principle_components = xr.concat(primary_pc, dim='event').where(primary_samples['timestamp'].dt.year.isin([2018, 2019, 2020, 2021, 2022])).mean(dim='event')
primary_mjo_composites = primary_samples.where(primary_samples['timestamp'].dt.year.isin([2018, 2019, 2020, 2021, 2022])).mean(dim='event')

# primary_principle_components = xr.concat(primary_pc, dim='event').mean(dim='event')
# primary_mjo_composites = primary_samples.mean(dim='event')

secondary_da = []
secondary_pc = []
for time in secondary_event_max_times:
    secondary_da.append(
        variables_filtered['Outgoing Longwave Radiation'].sel(
            time=slice(
                secondary_event_max_times.sel(time=time)-np.timedelta64(45, 'D'),
                secondary_event_max_times.sel(time=time)+np.timedelta64(45, 'D')
            )
        ).assign_coords(time=lag_days).rename(time='lag').expand_dims('event', axis=0).assign_coords(
            timestamp=("event", [secondary_event_max_times.sel(time=time).values])
        )
    )
    secondary_pc.append(
        principle_components.sel(
            time=slice(
                secondary_event_max_times.sel(time=time)-np.timedelta64(45, 'D'),
                secondary_event_max_times.sel(time=time)+np.timedelta64(45, 'D')
            )
        ).assign_coords(time=lag_days).rename(time='lag').expand_dims('event', axis=0).assign_coords(
            timestamp=("event", [secondary_event_max_times.sel(time=time).values])
        )
    )

secondary_samples = xr.concat(secondary_da, dim='event')
secondary_principle_components = xr.concat(secondary_pc, dim='event').where(secondary_samples['timestamp'].dt.year.isin([2018, 2019, 2020, 2021, 2022])).mean(dim='event')
secondary_mjo_composites = secondary_samples.where(secondary_samples['timestamp'].dt.year.isin([2018, 2019, 2020, 2021, 2022])).mean(dim='event')

# secondary_principle_components = xr.concat(secondary_pc, dim='event').mean(dim='event')
# secondary_mjo_composites = secondary_samples.mean(dim='event')

### Statistical Significance Testing

In [ ]:
from scipy.stats import ttest_1samp

primary_tstat, primary_pvals = xr.apply_ufunc(
    ttest_1samp,
    primary_samples,  # data
    0.0,              # test mean
    input_core_dims=[['event'], []],
    output_core_dims=[[], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float],
)

secondary_tstat, secondary_pvals = xr.apply_ufunc(
    ttest_1samp,
    secondary_samples,  # data
    0.0,              # test mean
    input_core_dims=[['event'], []],
    output_core_dims=[[], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float],
)

## Plot Composites

### Plot horizontal structures

In [ ]:
levels = np.arange(-30, 32.5, 2.5)

plt.style.use('default')
plt.rcParams.update({'font.size':12})
fig = plt.figure(figsize=(18,6))
gs = GridSpec(1, 3, figure=fig, width_ratios=[100,100,3])
gs.update(wspace=0.1)

cmap = plt.get_cmap('coolwarm').copy()
cmap.set_bad(color='white')

axes = [
    fig.add_subplot(gs[0]),
    fig.add_subplot(gs[1]),
]

cbar_axis = fig.add_subplot(gs[2])

axes[0].set_title('Secondary MJO Events', fontsize=18)
im = axes[0].contourf(
    secondary_mjo_composites.lon,
    secondary_mjo_composites.lag,
    secondary_mjo_composites.where(secondary_pvals < 0.05, other=np.nan).sel(lat=slice(-10,10)).mean(dim='lat'),
    # levels=np.arange(-30,35,5),
    levels = [-0.1, 0.1],
    cmap=cmap,
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)
axes[0].contour(
    secondary_mjo_composites.lon,
    secondary_mjo_composites.lag,
    secondary_mjo_composites.sel(lat=slice(-10,10)).mean(dim='lat'),
    # levels=im.levels[im.levels != 0],
    levels = levels[levels != 0],
    colors='k',
    linewidths=1
)
fig.colorbar(im, cax=cbar_axis, orientation='vertical', label=r'W m$^{-2}$')

axes[1].set_title('Primary MJO Events', fontsize=18)
im = axes[1].contourf(
    primary_mjo_composites.lon,
    primary_mjo_composites.lag,
    primary_mjo_composites.where(primary_pvals < 0.05, other=np.nan).sel(lat=slice(-10,10)).mean(dim='lat'),
    # levels=np.arange(-30,35,5),
    levels = [-0.1, 0.1],
    cmap=cmap,
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='both'
)
axes[1].contour(
    primary_mjo_composites.lon,
    primary_mjo_composites.lag,
    primary_mjo_composites.sel(lat=slice(-10,10)).mean(dim='lat'),
    # levels=im.levels[im.levels != 0],
    levels = levels[levels != 0],
    colors='k',
    linewidths=1
)

axes[0].set_ylabel('Lag (days)')
axes[1].set_yticklabels('')

for axis in axes:
    axis.set_xticks(np.arange(0, 390, 30), labels=tick_labeller(np.arange(0, 390, 30), 'lon'))
    axis.set_xlabel('Longitude')
    axis.axhline(y=0, color='k', lw=1, ls='-')
    axis.set_yticks(np.arange(-45,50,5))

plt.show()

### Plot lag day evolution

In [ ]:
# Set plotting parameters
output_directory = "output/mjo-compositing/"
plt.style.use('default')
plt.rcParams.update({'font.size':12})
coastline_width = 1

levels = np.arange(-30, 35, 5)

fig = plt.figure(figsize=(16,8))
gs = GridSpec(5, 2, figure=fig)
gs.update(hspace=0.1, wspace=0.25)

proj = ccrs.PlateCarree(central_longitude=-205)
data_crs = ccrs.PlateCarree()

axes = [
    [
        fig.add_subplot(gs[0,0], projection=proj),
        fig.add_subplot(gs[1,0], projection=proj),
        fig.add_subplot(gs[2,0], projection=proj),
        fig.add_subplot(gs[3,0], projection=proj),
        fig.add_subplot(gs[4,0], projection=proj)
    ],
    [
        fig.add_subplot(gs[0,1], projection=proj),
        fig.add_subplot(gs[1,1], projection=proj),
        fig.add_subplot(gs[2,1], projection=proj),
        fig.add_subplot(gs[3,1], projection=proj),
        fig.add_subplot(gs[4,1], projection=proj)
    ]
]

# Add cyclic point
cdata_secondary = xarray_utils.add_cyclic_point(
    secondary_samples.mean(dim='event'),
    dim='lon'
)

cdata_secondary_pvals = xarray_utils.add_cyclic_point(
    secondary_pvals,
    dim='lon'
)

cdata_primary = xarray_utils.add_cyclic_point(
    primary_samples.mean(dim='event'),
    dim='lon'
)

cdata_primary_pvals = xarray_utils.add_cyclic_point(
    primary_pvals,
    dim='lon'
)

for index, lag in enumerate([-25,-15,-5,5,15]):
    axes[0][index].contourf(
        cdata_secondary.lon,
        cdata_secondary.lat,
        cdata_secondary.where(cdata_secondary_pvals < 0.05, 1, np.nan).sel(lag=lag),
        transform=data_crs,
        cmap=cmap,
        norm=mcolors.CenteredNorm(),
        levels=[-1,1],
        extend='both'
    )
    axes[0][index].contour(
        cdata_secondary.lon,
        cdata_secondary.lat,
        cdata_secondary.sel(lag=lag),
        transform=data_crs,
        levels=levels[levels != 0],
        colors='k',
    )

    axes[1][index].contourf(
        cdata_primary.lon,
        cdata_primary.lat,
        cdata_primary.where(cdata_primary_pvals < 0.05, 1, np.nan).sel(lag=lag),
        transform=data_crs,
        cmap=cmap,
        norm=mcolors.CenteredNorm(),
        levels=[-1,1],
        extend='both'
    )
    axes[1][index].contour(
        cdata_primary.lon,
        cdata_primary.lat,
        cdata_primary.sel(lag=lag),
        transform=data_crs,
        levels=levels[levels != 0],
        colors='k',
    )

    for sub_axis_index in [0, 1]:
        axes[sub_axis_index][index].text(
            s=f"Day {lag}", x=330, y=-20, 
            ha='right', va='bottom',
            color='k', 
            fontsize=14, 
            zorder=10, 
            transform=data_crs,
            bbox=dict(facecolor='white', alpha=0.8)
        )
        axes[sub_axis_index][index].set_aspect('equal')
        axes[sub_axis_index][index].set_xlabel('')
        axes[sub_axis_index][index].add_feature(cf.COASTLINE, lw=coastline_width)

        gl = axes[sub_axis_index][index].gridlines(
            crs=proj,
            draw_labels=True,
            linewidth=1,
            # color="gray",
            alpha=0.75,
            linestyle="-",
            zorder=15
        )
        gl.right_labels = False
        gl.top_labels = False
        gl.xlocator = mticker.FixedLocator(np.arange(-180,180,30))
        # gl.xlocator = LongitudeLocator(30)
        gl.xformatter = LongitudeFormatter()
        gl.xlabel_style = {'fontsize':12}
        gl.ylocator = mticker.FixedLocator(np.arange(-40,40,10))
        gl.yformatter = LatitudeFormatter()
        gl.ylabel_style = {'fontsize':12}

plt.show()

### Plot composite PC time series

In [ ]:
point_marker = ['o', 'D', 's']
point_colors = ['k', 'k', 'k']
point_facecolors = ['none', 'k', 'k']
point_sizes = [15, 15, 15]
point_days = [-45, 0, 45]

# Configure plot
plt.style.use('default')
[fig, axes] = plt.subplots(1, 2, figsize=(12,6))

axes[0].plot(
    secondary_principle_components.sel(mode=1),
    secondary_principle_components.sel(mode=2),
    color='k',
    linestyle='-',
    marker="o",
    markerfacecolor='none',
    ms=4,
)

axes[1].plot(
    primary_principle_components.sel(mode=1),
    primary_principle_components.sel(mode=2),
    color='k',
    linestyle='-',
    marker="o",
    markerfacecolor='none',
    ms=4,
)

for point in range(3):
    axes[0].plot(
        secondary_principle_components.sel(mode=1, lag=point_days[point]),
        secondary_principle_components.sel(mode=2, lag=point_days[point]),
        color=point_colors[point],
        marker=point_marker[point],
        markerfacecolor=point_facecolors[point],
        ls='',
        ms=point_sizes[point]
    )

    axes[1].plot(
        primary_principle_components.sel(mode=1, lag=point_days[point]),
        primary_principle_components.sel(mode=2, lag=point_days[point]),
        color=point_colors[point],
        marker=point_marker[point],
        markerfacecolor=point_facecolors[point],
        ls='',
        ms=point_sizes[point]
    )

for index, axis in enumerate(axes):

    # Add phase regions overlay
    circle1 = plt.Circle((0, 0), 0.3, color="#bcbcbc", fill=False, lw=1, zorder=10)
    circle2 = plt.Circle((0, 0), 0.4, color="k", fill=False, lw=1.5, zorder=10)
    circle3 = plt.Circle((0, 0), 0.5, color="#bcbcbc", fill=False, lw=1, zorder=10)
    axis.add_patch(circle1)
    axis.add_patch(circle2)
    axis.add_patch(circle3)

    axis.axhline(y=0, color="k", lw=1, ls="-")
    axis.axvline(x=0, color="k", lw=1, ls="-")

    # Add lines to differentiate the phases
    axis.plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    axis.plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    axis.plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    axis.plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )

    axis.set_xlim(-2,2)
    axis.set_ylim(-2,2)

    # # Add phase labels
    axis.text(
        1.9, 0.1,
        f'Category A',
        horizontalalignment='right',
        verticalalignment='center',
        fontsize=12
    )
    axis.text(
        0, 1.8,
        f'Category B',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )
    axis.text(
        -1.9, .1,
        f'Category C',
        horizontalalignment='left',
        verticalalignment='center',
        fontsize=12
    )
    axis.text(
        0, -1.8,
        f'Category D',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )

    axis.spines['left'].set_position('zero')
    axis.spines['bottom'].set_position('zero')

    # Hide the top and right spines
    axis.spines['right'].set_color('none')
    axis.spines['top'].set_color('none')

    # Ensure tick marks follow the spines to the center
    axis.xaxis.set_ticks_position('bottom')
    axis.yaxis.set_ticks_position('left')

    axis.set_aspect("equal")
plt.tight_layout()

## Collapse events

In [ ]:
# --- INPUT ---
# categories: your daily xarray DataArray of regime labels
# categories.time: datetime64 coordinate

# 1. Extract arrays
arr   = categories.values.astype(str)      # ['A','A','B',...]
times = categories.time.values             # datetime64 array

# 2. Find change points (where the category changes)
change_points = np.where(arr[1:] != arr[:-1])[0] + 1

# 3. Collapse both categories AND timestamps
collapsed_categories = np.concatenate(([arr[0]],   arr[change_points]))
collapsed_times      = np.concatenate(([times[0]], times[change_points]))

# 4. Build collapsed DataArray with timestamps preserved
collapsed_da = xr.DataArray(
    data=collapsed_categories,
    dims=["time"],
    coords={"time": collapsed_times}
)

# 5. Print collapsed string for each year
for year, group in collapsed_da.groupby("time.year"):
    s = "".join(group.values)
    print(year, s)


In [ ]:
variables_Subset